In [1]:
from __future__ import annotations

import argparse
import copy
import json
from pathlib import Path
from typing import Any

import numpy as np

In [2]:
from histra.io.hr_loader import load_model
from histra.solver.solve import solve_static_nonlinear

from histra.tools.benchmark_csharp_sqlite import run_benchmark
from histra.tools.run_vert_live import run_vert_live

In [3]:
def json_value(value: Any) -> Any:
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    return value

In [4]:
model_path = Path("my_model/new_model.hrx")
model = load_model(model_path)

In [5]:
print(f"DOFs: {model.gdl}")

DOFs: 210


In [6]:
for key, analysis in sorted(model.collections.analyses.items()):
    print(
        f"key={key:<4} "
        f"name={analysis.name!r:<30} "
        f"integration={analysis.integration_method:<12} "
        f"method={analysis.method:<40} "
        f"combination={analysis.load_combination_key:<4} "
        f"initial_analysis={analysis.initial_analysis_key}"
    )

key=1    name='Vert'                         integration=LoadControl  method=ModifiedRegulaFalsiLineSearch            combination=6    initial_analysis=-100
key=21   name='Modal_0'                      integration=LoadControl  method=StandardNewtonRaphson                    combination=6    initial_analysis=1
key=22   name='LiveLoad_1'                   integration=ArcLength    method=StandardInitialInterpolatedLineSearch    combination=15   initial_analysis=1
key=23   name='scour_1'                      integration=LoadControl  method=ModifiedRegulaFalsiLineSearch            combination=6    initial_analysis=1
key=24   name='scour_2'                      integration=LoadControl  method=ModifiedRegulaFalsiLineSearch            combination=6    initial_analysis=23
key=25   name='LiveLoad_2'                   integration=ArcLength    method=StandardInitialInterpolatedLineSearch    combination=15   initial_analysis=24
key=26   name='scour_3'                      integration=LoadControl  m

In [7]:
analysis_to_run_key = 22
analysis_to_run = copy.deepcopy(model.collections.analyses[analysis_to_run_key])

combination = 1

In [8]:
print(f"Analysis: {analysis_to_run.key} — {analysis_to_run.name}")
print(f"Integration: {analysis_to_run.integration_method}")
print(f"Method: {analysis_to_run.method}")
# print(f"Combination row: {args.combination}")
print(f"Initial analysis: {analysis_to_run.initial_analysis_key}")
print()

Analysis: 22 — LiveLoad_1
Integration: ArcLength
Method: StandardInitialInterpolatedLineSearch
Initial analysis: 1



In [ ]:
def run_vert_live(
    hrx=model_path,
    output_dir= Path("python-results"),
    *,
    vert_selector: str | None = None,
    live_selector: str | None = None,
    combination: int = 1,
    echo_log: bool = True,
) -> dict[str, Any]:


run_benchmark(
    hrx=model_path,
    results=Path("my_model/new_model.Results"),
    analysis_key= analysis_to_run.key,
    combination= combination,
    selected_dofs= [10],
    echo_solver_log=True,
)

code, steps = solve_static_nonlinear(
    model,
    analysis_to_run,
    combination,
    on_log=lambda message: print(message, flush=True),
    # results_path=Path("my_model/new_model.Results"),
)

In [ ]:
committed = [row for row in steps if row["status"] == "OK"]
failed = [row for row in steps if row["status"] != "OK"]

In [ ]:
steps

In [ ]:
output = {
    # "model": str(args.hrx.resolve()),
    "analysis_key": analysis_to_run.key,
    "analysis_name": analysis_to_run.name,
    "combination": combination,
    "exit_code": code,
    "committed_steps": len(committed),
    "failed_steps": len(failed),
    "steps": [
        {key: json_value(value) for key, value in row.items()}
        for row in steps
    ],
}
output

In [ ]:
np.abs(np.array(output['steps'][-1]['u'])).max()

In [ ]:
# args.output.write_text(
#     json.dumps(output, indent=2, allow_nan=False),
#     encoding="utf-8",
# )

In [ ]:
print()
print(f"Exit code: {code}")
print(f"Committed steps: {len(committed)}")
if committed:
    last = committed[-1]
    print(f"Last committed step: {last['step']}")
    print(f"Last load multiplier: {last['load_factor']:.12g}")
    print(f"Last monitored displacement: {last['displacement']:.12g}")
    print(f"Iterations: {last['iterations']}")
if failed:
    print(f"Stopped during step: {failed[-1]['step']}")

# print(f"Results written to: {args.output}")

In [ ]:
run_benchmark(
    hrx=model_path,
    results=Path("my_model/new_model.Results"),
    analysis_key= analysis_to_run.key,
    combination= combination,
    selected_dofs= [10],
    echo_solver_log=True,
)